# Colab Drive RAG Run

Colab에서 RAG 실험을 돌릴 때 사용하는 노트북입니다. Drive에 데이터를 두고 공유하거나, HuggingFace embedding/reranker/LLM처럼 자원이 더 필요한 옵션을 실험할 때 사용합니다.
로컬 노트북과 같은 pipeline을 사용하되, 입력 데이터와 산출물 경로만 Drive 기준으로 바꿉니다.

## 1. Drive 연결

원본 문서, 평가 질문, 실험 결과를 Google Drive에 둘 수 있도록 Drive를 mount합니다.
팀원이 같은 경로를 쓰면 산출물 공유와 백업이 쉬워집니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 저장소 준비

`REPO_URL`은 GitHub 원격 저장소가 준비된 뒤 실제 주소로 바꿉니다.
`DRIVE_ROOT`는 원본 문서와 실험 결과를 저장할 Drive 작업 폴더입니다.

In [ ]:
REPO_URL = "<repo-url>"  # GitHub 저장소 주소로 변경 필수
REPO_DIR = "/content/codeit-rag-project"
DRIVE_ROOT = "/content/drive/MyDrive/codeit_rag_project"

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -r requirements.txt

## 3. Drive 폴더 준비

Drive에는 원본 문서, 평가 질문, 실험 결과, 백업을 분리해서 둡니다.
원본 문서는 `data/raw_docs`, 평가 질문은 `data/eval_questions.csv`에 둔다고 가정합니다.

In [ ]:
from pathlib import Path
drive_root = Path(DRIVE_ROOT)
for subdir in ["data/raw_docs", "experiments", "backups", "reports"]:
    (drive_root / subdir).mkdir(parents=True, exist_ok=True)

print(drive_root)

## 4. RAG config 경로

Colab에서는 기존 config를 복사해서 Drive 경로로 수정하는 것을 권장합니다.
`RAG_CONFIG`는 repo에 있는 기준 config이고, `COLAB_CONFIG`는 Drive 경로가 반영된 실행 config입니다.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    """현재 노트북 위치에서 프로젝트 루트 디렉터리를 찾습니다."""
    markers = ("AGENTS.md", "configs", "scripts", "src")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(
        "프로젝트 루트를 찾지 못했습니다. "
        "git clone 후 저장소 디렉터리에서 실행 중인지 확인해 주세요."
    )


PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
os.chdir(PROJECT_ROOT)
PYTHON = shlex.quote(sys.executable)

RAG_CONFIG = Path("configs/experiments/rag/rag_keyword.yaml")
# VM/Colab에서 실제 문서 실험 시 rag_langchain.yaml로 변경COLAB_CONFIG = Path("configs/experiments/rag/rag_colab_drive.yaml")
QUESTION = "예산은 얼마야?"
OUTPUT_DIR = Path(DRIVE_ROOT) / "experiments" / "rag_colab_drive"

print(f"project root: {PROJECT_ROOT}")
print(f"python: {sys.executable}")


## OpenAI API key 선택 설정

OpenAI 사용 여부는 노트북 변수가 아니라 config의 `rag.embedding.provider` 또는 `rag.answerer.provider`로 결정합니다.
기본 config는 local provider라 비용이 발생하지 않습니다. OpenAI provider config를 선택했을 때만 아래 환경변수가 필요합니다.

In [ ]:
# OpenAI provider config를 사용할 때만 아래 두 줄의 주석을 해제합니다.
# import getpass
# os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("OPENAI_API_KEY 설정 여부:", bool(os.environ.get("OPENAI_API_KEY")))

## 5. Colab config 작성 예시

아래 셀은 예시입니다. 실제 문서와 평가 질문 파일이 Drive에 준비된 뒤 사용합니다.
retriever나 splitter 옵션을 바꾸고 싶으면 이 YAML의 `rag` 블록을 수정합니다.

In [ ]:
colab_config_text = f"""
experiment:
  name: rag_colab_drive
  author: team
  seed: 42
  contract_version: rag-v0.2

paths:
  raw_docs_dir: {DRIVE_ROOT}/data/raw_docs
  output_dir: {DRIVE_ROOT}/experiments/rag_colab_drive

artifact_policy:
  run_id:
  on_existing: overwrite

rag:
  engine: langchain
  loader:
    file_types: [txt, pdf, docx, hwpx, hwp]
  splitter:
    type: recursive_character
    chunk_size: 500
    chunk_overlap: 80
  checkpoint:
    enabled: true
    resume: true
  embedding:
    provider: local
    model_name: hashing-char-ngram-v1
    dimension: 64
    device: auto
    normalize: true
  vector_store:
    type: memory
    path:
    collection_name: rag_colab_drive
  retriever:
    method: semantic
    top_k: 3
    score_threshold: 0.0
  reranker:
    enabled: false
    provider: huggingface
    model_name:
    top_k: 3
  answerer:
    mode: extractive
    provider: local
    model_name:
    fallback_message: 문서에서 확인하지 못했습니다.

evaluation:
  questions_path: {DRIVE_ROOT}/data/eval_questions.csv

metric:
  monitor: retrieval_hit_rate
  mode: max

backup:
  enabled: true
  on_finish: true
  on_failure: true
  backup_dir: {DRIVE_ROOT}/backups/rag_colab_drive
  include_logs: true
  include_checkpoints: true
"""
COLAB_CONFIG.write_text(colab_config_text.strip() + "\n", encoding="utf-8")
print(COLAB_CONFIG)

In [ ]:
def run(command: str) -> None:
    print(f"$ {command}")
    result = __import__("subprocess").run(command, shell=True, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(command)

def show_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def show_csv(path, n=5):
    path = Path(path)
    return pd.read_csv(path).head(n) if path.exists() else pd.DataFrame()

def show_jsonl(path, n=3):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()][:n]

def display_metrics(path):
    metrics = show_json(path)
    if not metrics:
        return pd.DataFrame()
    return pd.DataFrame([{"metric": key, "value": value} for key, value in metrics.items()])

def display_answers(path, n=5):
    rows = show_jsonl(path, n=n)
    summary = []
    for row in rows:
        citations = row.get("citations", [])
        citation_text = "; ".join(
            f"{item.get('source_path', '')}:{item.get('chunk_id', '')}"
            for item in citations
        )
        summary.append({
            "question": row.get("question"),
            "status": row.get("status"),
            "answer": row.get("answer"),
            "citations": citation_text,
        })
    return pd.DataFrame(summary)

def display_failure_tables(output_dir, n=5):
    output_dir = Path(output_dir)
    for title in ["bad_retrievals", "unsupported_answers", "failed_questions"]:
        print(f"## {title}")
        display(show_csv(output_dir / f"{title}.csv", n=n))

## 6. 실행 전 점검

In [ ]:
run(f"{PYTHON} scripts/check_rag_pipeline.py --config {COLAB_CONFIG} --project-root .")

## 7. RAG 실행

config 점검이 통과하면 ingest, retrieve, evaluate를 순서대로 실행합니다.
중간에 실패하면 먼저 Drive 경로와 평가 질문 CSV가 맞는지 확인합니다.

In [ ]:
run(f"{PYTHON} scripts/run_rag_ingest.py --config {COLAB_CONFIG} --project-root .")
run(f"{PYTHON} scripts/run_rag_retrieve.py --config {COLAB_CONFIG} --project-root . --question {shlex.quote(QUESTION)}")
run(f"{PYTHON} scripts/run_rag_chat.py --config {COLAB_CONFIG} --project-root . --evaluate")

## 8. 결과 확인

결과는 JSON 원본이 아니라 표 형태로 확인합니다.
metric, 실패 질문, 답변과 citation을 함께 봐야 retrieval 문제와 answerer 문제를 구분할 수 있습니다.

In [ ]:
display_metrics(OUTPUT_DIR / "metrics.json")

In [ ]:
display_failure_tables(OUTPUT_DIR)
display_answers(OUTPUT_DIR / "answers.jsonl")

## 9. 공유할 것

- 사용한 config 경로
- Drive 결과 폴더
- `metrics.json` 요약
- 성공 질문 1개와 실패 질문 1개
- 답변과 citation 캡처